# Homework 4 — Portfolio Q-learning desk (Module 4 framework)

**Salih Tutun, PhD** · DAT 5668 · Module 4

**Individual.** Due after **Session 8**.

Open this file in **Google Colab** from Canvas (or run it locally). When you are done, download the `.ipynb` and upload it on the Canvas assignment.

Session 8 practiced tabular Q-learning on an **inventory** desk. This homework keeps the **same Module 4 framework** on a **new finance desk**: a **SPY ETF portfolio** where each day you may **Buy**, **Hold**, or **Sell**.

You will:

1. **DEFINE** the MDP in your own words  
2. **Read every commented line** of the loader and pipeline  
3. **Design rewards** and say why (discussion after the part)  
4. **Write** `epsilon_greedy` + `q_update`  
5. **CHANGE ME** knobs (`α`, `ε`, `γ`, episodes) and comment what moved  
6. **Break** the Bullish reward and discuss with a CIO voice  
7. Leave a **framework** line + human override  

**Data:** `portfolio_spy_daily.csv` — real SPY daily history (Yahoo Finance), **≥ 2000** trading days, **≥ 6** numeric features.

**Deliverable:** every `<<< FILL` / discussion box completed and `review_hw4()` prints **PASS**.

---

## How to run this in Colab

1. Open from Canvas **Homework 4 — Open in Colab**, or use [this Colab file](https://colab.research.google.com/github/salihtutun/Agentic-AI_Autonomous_Decision_Making/blob/main/Students/Homework/HW4-Portfolio-Q-Learning.ipynb).
2. Run cells **top to bottom** (Shift+Enter). Do not skip the data cell.
3. After each part, fill the **YOUR TAKE** cell before moving on.
4. Change at least one knob marked `<<< CHANGE ME` and write what happened.
5. Run `review_hw4()` until it prints **PASS**.
6. File → Download → `.ipynb`. Upload on **Homework 4 — Portfolio Q-learning**.

If a cell says `NameError`, you skipped a cell above it. Run from the top.

## Comment + discuss rules (read once)

- Code cells already explain **each line**. Keep those comments when you edit.
- After Parts 1–6, a **YOUR TAKE** cell asks for *your* idea (not a copy of the prompt).
- When you change a parameter, add a short comment on that same line or in the TAKE cell.


## Business story — SPY portfolio desk (finance)

You support a portfolio desk that trades the **SPY** ETF (U.S. large-cap equity exposure).

Each morning the desk observes a **market regime** from 10-day momentum, then chooses:

- **Buy** — add SPY exposure (deploy cash into the market)  
- **Hold** — keep the current book  
- **Sell** — reduce SPY exposure (raise cash / take profit / cut risk)  

| State (from `Momentum_10d`) | Meaning |
| --- | --- |
| Bearish Market | 10-day return $< -1\%$ |
| Neutral Market | between $-1\%$ and $+1\%$ |
| Bullish Market | 10-day return $> +1\%$ |

Dataset columns (6+ features):

`Close`, `Daily_Return`, `Volatility_20d`, `Momentum_10d`, `Volume`, `RSI_14`, `Next_Day_Return`

```text
load SPY days
  → observe market regime s
  → decide a with ε-greedy on Q   (Buy / Hold / Sell)
  → reward r from your P&L-style table
  → next state s' = tomorrow's regime
  → q_update
  → repeat
```

A chatbot that never updates $Q$ is **not** doing Module 4. A policy that **Sells into a Bullish** regime is leaving money on the table — or fighting the tape.


## Part 1 — DEFINE (before code)

Fill the next cell for the **portfolio** desk. Do not paste inventory wording.

| Piece | Your job |
| --- | --- |
| State $s$ | What market signal do you observe before trading? |
| Actions $\mathcal{A}$ | List Buy, Hold, Sell |
| Reward $r$ | What money / risk score does the table represent? |
| Next state $s'$ | Where does tomorrow’s regime come from? |
| Not in $s$ | Name one document/exception a person still owns |


In [ ]:
# ============================================================
# PART 1 — FILL (DEFINE in words)
# Each line below is one piece of the Module 4 MDP vocabulary.
# ============================================================

# s: what the desk can measure BEFORE choosing Buy / Hold / Sell
state_s = ""  # <<< FILL: market regime from momentum (your words)

# A: the three legal portfolio actions for this homework
actions = []  # <<< FILL: ["Buy", "Hold", "Sell"]

# r: the scalar the reward table is trying to stand in for (P&L / risk)
reward_r = ""  # <<< FILL: money / risk language

# s': tomorrow's regime comes from the next row of the SPY CSV
next_state = ""  # <<< FILL: next trading day's regime

# exception path: documents are NOT a number inside s
not_in_state = ""  # <<< FILL: compliance halt, CIO email, earnings memo, ...

# Print so you can read your DEFINE out loud before coding
print("s:", state_s)
print("A:", actions)
print("r:", reward_r)
print("s':", next_state)
print("not in s:", not_in_state)


In [ ]:
# ============================================================
# PART 1 — YOUR TAKE (discuss in your own words)
# ============================================================

# Why is Buy/Hold/Sell a better fit for a *numeric* Q-table than for a pure LLM agent?
part1_discussion = ""  # <<< FILL ≥ 80 characters — your idea, not the prompt

# Optional: if you disagree with using only Momentum_10d for the state, say what you would add.
part1_alternative_state = ""  # <<< FILL or write "I keep Momentum_10d because ..."

print("Part 1 discussion:", part1_discussion or "(FILL)")
print("Alternative state idea:", part1_alternative_state or "(FILL)")


## Part 2 — LOAD the SPY data (every line commented)

Run this cell so the homework can see the portfolio CSV.  
Loads **SPY** days and builds regimes from `Momentum_10d`.  
Read the comments. You may **CHANGE ME** the regime thresholds — then explain in YOUR TAKE.


In [ ]:
# ============================================================
# PART 2 — LOAD (SPY portfolio data)
# Read every comment. Change thresholds only after you understand them.
# ============================================================

from pathlib import Path          # Path joins folders the same way on Windows / Colab
from collections import Counter   # Counter tallies how many days sit in each regime
import csv                        # csv.DictReader reads one market day per row
import random                     # random.Random gives a fixed ε-greedy stream later

# Optional Colab Drive path (same folder layout as other DAT 5668 notebooks)
DRIVE_DATA_FOLDER = "/content/drive/MyDrive/DAT5668/data"
DATA_NAME = "portfolio_spy_daily.csv"         # homework CSV name on Drive / GitHub / assets
RNG = random.Random(7)            # seed 7 → everyone shares the same explore stream


def course_data(name: str) -> Path:
    """Find the CSV on Drive, local assets, or GitHub raw fallback."""
    candidates = []                                   # list of places we will look
    if DRIVE_DATA_FOLDER:                             # Colab + shared course Drive
        candidates.append(Path(DRIVE_DATA_FOLDER) / name)
    here = Path.cwd()                                 # wherever the notebook is running
    for base in (                                     # local / repo / Colab layouts
        Path("data"),
        Path("assets/data"),
        Path("../assets/data"),
        Path("../../assets/data"),
        here / "assets" / "data",
        here.parent / "assets" / "data",
        Path("/content/data"),
    ):
        candidates.append(base / name)                # try this folder + filename
    for path in candidates:                           # first existing path wins
        if path.exists():
            return path
    import urllib.request                             # only if nothing local worked

    dest = Path("/content/data") if Path("/content").exists() else Path("data")
    dest.mkdir(parents=True, exist_ok=True)           # make a download folder
    out = dest / name                                 # local file we will write
    url = (
        "https://raw.githubusercontent.com/salihtutun/"
        "Agentic-AI_Autonomous_Decision_Making/main/assets/data/" + name
    )
    urllib.request.urlretrieve(url, out)              # pull from public GitHub
    return out                                        # return the downloaded path


# <<< CHANGE ME: regime cutoffs (try ±0.005 or ±0.02 after first PASS)
BEARISH_CUT = -0.01   # momentum below this → Bearish Market
BULLISH_CUT = 0.01    # momentum above this → Bullish Market
# comment your change here if you edit the cuts:
# cutoff_comment = "I left ±1% ..."


def market_regime(momentum_10d: float) -> str:
    """Map one number (10-day return) into three discrete states for tabular Q."""
    if momentum_10d < BEARISH_CUT:                    # weak recent tape
        return "Bearish Market"
    if momentum_10d <= BULLISH_CUT:                   # quiet / sideways tape
        return "Neutral Market"
    return "Bullish Market"                           # strong recent tape


ROWS = []                                             # will hold one dict per trading day
path = course_data(DATA_NAME)                         # resolve CSV path once
with path.open(encoding="utf-8") as handle:           # open text file
    for row in csv.DictReader(handle):                # each CSV row → dict of strings
        mom = float(row["Momentum_10d"])              # 10-day return used for the state
        ROWS.append(                                  # store typed fields we need later
            {
                "Date": row["Date"],                  # calendar day
                "Close": float(row["Close"]),         # SPY close
                "Daily_Return": float(row["Daily_Return"]),
                "Volatility_20d": float(row["Volatility_20d"]),
                "Momentum_10d": mom,                  # state signal
                "Volume": int(float(row["Volume"])),  # shares traded
                "RSI_14": float(row["RSI_14"]),       # 14-day RSI
                "Next_Day_Return": float(row["Next_Day_Return"]),
                "Action_Taken": (row.get("Action_Taken") or "").strip(),  # heuristic label
                "State": market_regime(mom),          # discrete s for Q-learning
            }
        )

STATES = ["Bearish Market", "Neutral Market", "Bullish Market"]  # rows of the Q-table
ACTIONS = ["Buy", "Hold", "Sell"]                                 # columns of the Q-table
N_DAYS = len(ROWS)                                                # how many samples we loaded
FEATURE_COLS = [                                                  # ≥6 numeric features in the file
    "Close",
    "Daily_Return",
    "Volatility_20d",
    "Momentum_10d",
    "Volume",
    "RSI_14",
    "Next_Day_Return",
]

print("File:", path.name)
print("Trading days:", N_DAYS, "(need ≥ 2000)")
print("Features (", len(FEATURE_COLS), "):", FEATURE_COLS)
print("Date range:", ROWS[0]["Date"], "→", ROWS[-1]["Date"])
print("Regime mix:", dict(Counter(r["State"] for r in ROWS)))
print("Cuts used: bearish <", BEARISH_CUT, "| bullish >", BULLISH_CUT)
assert N_DAYS >= 2000, "Dataset too small — re-download portfolio_spy_daily.csv"
assert len(FEATURE_COLS) >= 6
print("Data ready. Next: design trading rewards, then write ε-greedy + Q-update.")


In [ ]:
# ============================================================
# PART 2 — YOUR TAKE
# ============================================================

# What did the regime mix tell you about this SPY sample? (bullish-heavy? balanced?)
part2_discussion = ""  # <<< FILL ≥ 80 characters

# If you changed BEARISH_CUT / BULLISH_CUT, what did you try and what moved?
part2_param_comment = ""  # <<< FILL (or "I kept ±1% because ...") ≥ 40 characters

print("Part 2 discussion:", part2_discussion or "(FILL)")
print("Cutoff comment:", part2_param_comment or "(FILL)")


## Part 3 — Design the portfolio reward table (earn money when the regime is right)

Here **you** set the table so a portfolio desk would agree:

| Constraint (must hold) | Why |
| --- | --- |
| `R(Bullish, Buy) > R(Bullish, Sell)` | Do not fight a rising market |
| `R(Bearish, Sell) > R(Bearish, Buy)` | Cut risk / raise cash in a falling tape |
| Neutral: `Hold` should be best (or tied) | Avoid churn when the signal is flat |

Fill every entry. Numbers are **your** design — then defend them in YOUR TAKE.


In [ ]:
# ============================================================
# PART 3 — FILL reward table + rationale
# Tip: start near Session 8 intuition, then tune magnitudes if you want.
# ============================================================

# REWARDS[(state, action)] = how "good" that trade feels in that regime
REWARDS = {
    ("Bearish Market", "Buy"): None,    # <<< FILL — buying weakness should not look best
    ("Bearish Market", "Hold"): None,   # <<< FILL
    ("Bearish Market", "Sell"): None,   # <<< FILL — often best in Bearish
    ("Neutral Market", "Buy"): None,    # <<< FILL
    ("Neutral Market", "Hold"): None,   # <<< FILL — best or tied in Neutral
    ("Neutral Market", "Sell"): None,   # <<< FILL
    ("Bullish Market", "Buy"): None,    # <<< FILL — often best in Bullish
    ("Bullish Market", "Hold"): None,   # <<< FILL
    ("Bullish Market", "Sell"): None,   # <<< FILL — must lose to Buy
}

# Say why Bullish and Bearish look the way they do (money / risk words)
reward_rationale = ""  # <<< FILL ≥ 60 characters


def _row_ok():
    """Small checker used by Part 3 printout and by review_hw4()."""
    for k, v in REWARDS.items():                      # every cell must be a number
        if v is None:
            return False, f"missing reward for {k}"
    if not (REWARDS[("Bullish Market", "Buy")] > REWARDS[("Bullish Market", "Sell")]):
        return False, "Bullish: Buy must beat Sell"
    if not (REWARDS[("Bearish Market", "Sell")] > REWARDS[("Bearish Market", "Buy")]):
        return False, "Bearish: Sell must beat Buy"
    neu = [REWARDS[("Neutral Market", a)] for a in ACTIONS]
    if REWARDS[("Neutral Market", "Hold")] < max(neu):
        return False, "Neutral: Hold should be best (or tied)"
    if len(reward_rationale.strip()) < 60:
        return False, "reward_rationale needs ≥ 60 characters"
    return True, "reward table OK"


ok, msg = _row_ok()                                   # run the local checks
print(msg)
for s in STATES:                                      # show one row per regime
    print(s, {a: REWARDS[(s, a)] for a in ACTIONS})


In [ ]:
# ============================================================
# PART 3 — YOUR TAKE
# ============================================================

# Defend one number you chose (e.g. why Bullish Buy is +2 not +10).
part3_discussion = ""  # <<< FILL ≥ 80 characters

# Would you ever reward Sell in Bullish for a *risk* reason? Why or why not?
part3_risk_note = ""  # <<< FILL ≥ 40 characters

print("Part 3 discussion:", part3_discussion or "(FILL)")
print("Risk note:", part3_risk_note or "(FILL)")


## Part 4 — PIPELINE: write `epsilon_greedy` + `q_update`

Same functions you practiced in Session 8 — now on consecutive **SPY** days.

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \big(r + \gamma \max_{a'} Q(s',a') - Q(s,a)\big)
$$

**CHANGE ME** after your first PASS: try a new `alpha` / `epsilon` / `gamma` / `n_episodes` and comment what moved in YOUR TAKE.


In [ ]:
# ============================================================
# PART 4 — FILL ε-greedy + Q-update, then train
# Knobs below are yours to change after the first PASS.
# ============================================================

# <<< CHANGE ME after first PASS (add a trailing comment when you change a value)
alpha = 0.3       # learning rate α — how hard we pull Q toward the TD target
epsilon = 0.5     # explore probability ε — random action vs greedy action
gamma = 0.7       # discount γ — how much tomorrow's max Q matters today
n_episodes = 40   # how many passes over the SPY day sequence

# Empty Q-table: every (state, action) starts at 0.0
Q = {s: {a: 0.0 for a in ACTIONS} for s in STATES}


def epsilon_greedy(state, Q, epsilon):
    """Decide: explore a random action with prob epsilon, else take argmax_a Q[state][a]."""
    # <<< FILL
    # if RNG.random() < epsilon: return RNG.choice(ACTIONS)
    # else: return max(Q[state], key=Q[state].get)
    return None


def q_update(Q, state, action, reward, next_state, alpha, gamma):
    """Update Q[state][action] in place with the one-step TD target."""
    # <<< FILL
    # old = Q[state][action]
    # td_target = reward + gamma * max(Q[next_state].values())
    # Q[state][action] = old + alpha * (td_target - old)
    return None


def train_q(rewards, seed=7):
    """
    Module 4 pipeline on consecutive SPY regime days.
    Uses a *local* RNG so the student ε-greedy fill and the trainer stay testable.
    """
    local_rng = random.Random(seed)                   # fixed explore stream for grading
    Q_local = {s: {a: 0.0 for a in ACTIONS} for s in STATES}  # fresh table each train
    for _ in range(n_episodes):                       # repeat passes over history
        for i in range(len(ROWS) - 1):                # day i → day i+1
            state = ROWS[i]["State"]                  # observe s
            if local_rng.random() < epsilon:          # explore?
                action = local_rng.choice(ACTIONS)    # random Buy/Hold/Sell
            else:
                action = max(Q_local[state], key=Q_local[state].get)  # greedy
            reward = rewards[(state, action)]         # table lookup for r
            next_state = ROWS[i + 1]["State"]         # environment gives s'
            old = Q_local[state][action]              # Q before the update
            # classic tabular Q-learning update (same picture as Session 7–8)
            Q_local[state][action] = old + alpha * (
                reward + gamma * max(Q_local[next_state].values()) - old
            )
    return Q_local                                    # learned Q-table


# Tiny hand check: ε=0 must pick argmax; one update with γ≈0.9 from zeros → 1.0
_hand_Q = {"Bullish Market": {"Buy": 0.0, "Hold": 0.0, "Sell": 0.0}}
try:
    a0 = epsilon_greedy("Bullish Market", _hand_Q, 0.0)   # must return "Buy" once filled
    q_update(_hand_Q, "Bullish Market", "Buy", 2.0, "Bullish Market", 0.5, 0.9)
    # 0 + 0.5*(2 + 0.9*0 - 0) = 1.0 when next-state max Q is still 0
    hand_ok = a0 == "Buy" and abs(_hand_Q["Bullish Market"]["Buy"] - 1.0) < 1e-9
except Exception as err:
    hand_ok = False
    print("NOT YET — fix epsilon_greedy / q_update:", err)

if hand_ok:
    Q = train_q(REWARDS)                              # train on the student's reward table
    policy = {s: max(Q[s], key=Q[s].get) for s in STATES}  # greedy policy π(s)
    agree = sum(1 for r in ROWS if policy[r["State"]] == r["Action_Taken"])
    print("Learned policy:", policy)
    print("Agree with heuristic Action_Taken:", agree, "/", N_DAYS, f"({agree / N_DAYS:.1%})")
    print(
        "CHECK 4 —",
        "PASS"
        if policy["Bullish Market"] == "Buy" and policy["Bearish Market"] == "Sell"
        else "LOOK — expected Bullish→Buy and Bearish→Sell with a sensible reward table",
    )
else:
    policy = None
    print("Fill epsilon_greedy and q_update, then re-run this cell.")


In [ ]:
# ============================================================
# PART 4 — YOUR TAKE (parameters + ideas)
# ============================================================

# Explain ε-greedy in one breath: when do you explore vs exploit on this desk?
part4_discussion = ""  # <<< FILL ≥ 80 characters

# Which knob did you change (α / ε / γ / n_episodes)? What moved in the policy or agree %?
part4_param_comment = ""  # <<< FILL ≥ 60 characters — name the old→new value

print("Part 4 discussion:", part4_discussion or "(FILL)")
print("Parameter comment:", part4_param_comment or "(FILL)")


## Part 5 — Break the reward on purpose (interpret the trading policy)

Copy your table into `BROKEN`, then **flip** the Bullish row so **Sell** looks best (and Buy looks bad). Retrain. Compare policies.

Write `NOTE` (≥ 100 characters) and use YOUR TAKE to discuss deploy / no-deploy.


In [ ]:
# ============================================================
# PART 5 — FILL broken rewards + NOTE
# ============================================================

BROKEN = dict(REWARDS)            # start from *your* table (shallow copy of keys)
# <<< FILL: make Bullish→Sell look best (example direction: Sell = +5, Buy = -1)
BROKEN[("Bullish Market", "Sell")] = None
BROKEN[("Bullish Market", "Buy")] = None

# CIO-facing sentence: what went wrong under the broken incentives?
NOTE = ""  # <<< FILL ≥ 100 characters


def _broken_ready():
    """Local checks for Part 5 before we trust policy_bad."""
    if None in (BROKEN[("Bullish Market", "Sell")], BROKEN[("Bullish Market", "Buy")]):
        return False, "Set both BROKEN Bullish Sell and Bullish Buy numbers"
    if not (BROKEN[("Bullish Market", "Sell")] > BROKEN[("Bullish Market", "Buy")]):
        return False, "Broken table must prefer Sell over Buy when Bullish"
    if len(NOTE.strip()) < 100:
        return False, "NOTE needs ≥ 100 characters"
    return True, "broken setup OK"


bok, bmsg = _broken_ready()
print(bmsg)
if bok and hand_ok:
    Q_bad = train_q(BROKEN, seed=7)                   # same pipeline, bad incentives
    policy_bad = {s: max(Q_bad[s], key=Q_bad[s].get) for s in STATES}
    print("Policy with YOUR rewards:", policy)
    print("Policy with BROKEN rewards:", policy_bad)
    print(
        "CHECK 5 —",
        "PASS" if policy_bad["Bullish Market"] == "Sell" else "LOOK — broken Bullish should learn Sell",
    )
else:
    policy_bad = None


In [ ]:
# ============================================================
# PART 5 — YOUR TAKE
# ============================================================

# If a teammate said "the model is fine, just trust Q", how would you answer using Part 5?
part5_discussion = ""  # <<< FILL ≥ 80 characters

print("Part 5 discussion:", part5_discussion or "(FILL)")


## Part 6 — Framework line (Module 4)

Name the family and one sentence you will keep. Also name a **human override** if a Q-row ever says **Sell on Bullish** (or Buy on Bearish) during a live week.


In [ ]:
# ============================================================
# PART 6 — FILL framework
# ============================================================

framework_family = ""  # <<< FILL: LLM / RL / hybrid
framework_line = ""    # <<< FILL ≥ 40 characters — say the loop in business words
human_override = ""    # <<< FILL ≥ 40 characters — who stops a bad Sell-on-Bullish?

print("Family:", framework_family or "(FILL)")
print("I will keep:", framework_line or "(FILL)")
print("Human override:", human_override or "(FILL)")


In [ ]:
# ============================================================
# PART 6 — YOUR TAKE
# ============================================================

# One sentence for your write-up: what will you keep from Module 4?
part6_discussion = ""  # <<< FILL ≥ 80 characters

print("Part 6 discussion:", part6_discussion or "(FILL)")


In [ ]:
# ============================================================
# REVIEW — run last. Must print PASS
# Checks DEFINE, rewards, functions, broken run, framework, AND discussion boxes.
# ============================================================

def review_hw4():
    """Student-facing checker for Homework 4 (portfolio + discussions)."""
    issues = []

    # ---- Part 1 DEFINE ----
    if len((state_s or "").strip()) < 12:
        issues.append("Part 1: state_s — describe the market regime / momentum state.")
    if set(a.lower() for a in (actions or [])) < {"buy", "hold", "sell"}:
        issues.append('Part 1: actions must include Buy, Hold, Sell.')
    if len((reward_r or "").strip()) < 10:
        issues.append("Part 1: reward_r — say what money/risk the table scores.")
    if len((next_state or "").strip()) < 10:
        issues.append("Part 1: next_state — tomorrow's market regime.")
    if len((not_in_state or "").strip()) < 8:
        issues.append("Part 1: not_in_state — name a document/exception path.")
    if len((part1_discussion or "").strip()) < 80:
        issues.append("Part 1 YOUR TAKE: part1_discussion needs ≥ 80 characters.")
    if len((part1_alternative_state or "").strip()) < 12:
        issues.append("Part 1 YOUR TAKE: part1_alternative_state — keep or propose a state idea.")

    # ---- Part 2 discussion ----
    if len((part2_discussion or "").strip()) < 80:
        issues.append("Part 2 YOUR TAKE: part2_discussion needs ≥ 80 characters.")
    if len((part2_param_comment or "").strip()) < 40:
        issues.append("Part 2 YOUR TAKE: part2_param_comment needs ≥ 40 characters.")

    # ---- Part 3 rewards ----
    ok3, msg3 = _row_ok()
    if not ok3:
        issues.append("Part 3: " + msg3)
    if len((part3_discussion or "").strip()) < 80:
        issues.append("Part 3 YOUR TAKE: part3_discussion needs ≥ 80 characters.")
    if len((part3_risk_note or "").strip()) < 40:
        issues.append("Part 3 YOUR TAKE: part3_risk_note needs ≥ 40 characters.")

    # ---- Part 4 functions ----
    try:
        probe = {s: {a: 0.0 for a in ACTIONS} for s in STATES}
        probe["Bullish Market"]["Buy"] = 1.0
        greedy = epsilon_greedy("Bullish Market", probe, 0.0)
        if greedy != "Buy":
            issues.append("Part 4: epsilon_greedy(..., epsilon=0) must return the argmax action.")
        demoQ = {"Bullish Market": {"Buy": 0.0, "Hold": 0.0, "Sell": 0.0}}
        q_update(demoQ, "Bullish Market", "Buy", 2.0, "Bullish Market", 0.5, 0.0)
        if abs(demoQ["Bullish Market"]["Buy"] - 1.0) > 1e-9:
            issues.append("Part 4: q_update math — with γ=0, α=0.5, r=2 from 0 should become 1.0.")
    except Exception as err:
        issues.append(f"Part 4: epsilon_greedy/q_update error — {err}")

    if policy is None:
        issues.append("Part 4: train a policy (Bullish→Buy, Bearish→Sell) before review.")
    else:
        if policy.get("Bullish Market") != "Buy":
            issues.append("Part 4: with your REWARDS, Bullish Market should prefer Buy.")
        if policy.get("Bearish Market") != "Sell":
            issues.append("Part 4: with your REWARDS, Bearish Market should prefer Sell.")
    if len((part4_discussion or "").strip()) < 80:
        issues.append("Part 4 YOUR TAKE: part4_discussion needs ≥ 80 characters.")
    if len((part4_param_comment or "").strip()) < 60:
        issues.append("Part 4 YOUR TAKE: part4_param_comment needs ≥ 60 characters (name the knob you changed).")

    # ---- Part 5 broken ----
    ok5, msg5 = _broken_ready()
    if not ok5:
        issues.append("Part 5: " + msg5)
    elif policy_bad is not None and policy_bad.get("Bullish Market") != "Sell":
        issues.append("Part 5: broken table should teach Bullish→Sell.")
    if len((part5_discussion or "").strip()) < 80:
        issues.append("Part 5 YOUR TAKE: part5_discussion needs ≥ 80 characters.")

    # ---- Part 6 framework ----
    fam = (framework_family or "").strip()
    if fam not in {"LLM", "RL", "hybrid"}:
        issues.append("Part 6: framework_family must be LLM, RL, or hybrid.")
    if fam == "LLM":
        issues.append("Part 6: a numeric portfolio Q-table is RL (or hybrid with a human override) — not pure LLM.")
    if len((framework_line or "").strip()) < 40:
        issues.append("Part 6: framework_line needs ≥ 40 characters.")
    if len((human_override or "").strip()) < 40:
        issues.append("Part 6: human_override needs ≥ 40 characters.")
    if len((part6_discussion or "").strip()) < 80:
        issues.append("Part 6 YOUR TAKE: part6_discussion needs ≥ 80 characters.")

    if N_DAYS < 2000:
        issues.append("Data: need ≥ 2000 trading days in portfolio_spy_daily.csv.")
    if len(FEATURE_COLS) < 6:
        issues.append("Data: need ≥ 6 feature columns.")

    if issues:
        print("NOT YET")
        for issue in issues:
            print("-", issue)
        return False

    print("PASS")
    print("Policy:", policy)
    print(fam + ":", framework_line)
    print("Nice work — discussions + knobs + Q-update are in place.")
    return True


review_hw4()


## Submit checklist

1. [ ] Part 1 DEFINE + YOUR TAKE  
2. [ ] Part 2 data loaded; cutoff comment filled (even if you kept ±1%)  
3. [ ] Part 3 rewards + discussion / risk note  
4. [ ] Part 4 `epsilon_greedy` + `q_update` → Bullish→Buy, Bearish→Sell  
5. [ ] Part 4: changed at least one knob and explained it  
6. [ ] Part 5 broken experiment + NOTE + discussion  
7. [ ] Part 6 framework + YOUR TAKE  
8. [ ] `review_hw4()` prints **PASS**

**Turn in:** this completed notebook on Canvas (**Homework 4 — Portfolio Q-learning**).
